# EXAONE 4.0 32B Colab Pro Server

This notebook runs `LGAI-EXAONE/EXAONE-4.0-32B` in non-reasoning mode on a Colab Pro A100 40GB GPU. The model uses NF4 4-bit quantization and exposes an OpenAI-compatible API for the local `K-Safety Law RAG` CLI/Web UI.

## Flow
1. Select a Colab Pro A100 GPU runtime. T4 and L4 runtimes are intentionally rejected.
2. Run cells from top to bottom. The model is downloaded directly from the Hugging Face Hub into this Colab instance (no Google Drive, no local upload).
3. Log in to Hugging Face when prompted (accept the model license on the model page first if required).
4. Copy the ngrok URL printed at the end.
5. Set local `.env` with `LLM_API_BASE=<ngrok-url>/v1`.
6. Set local `.env` with `LLM_MODEL=LGAI-EXAONE/EXAONE-4.0-32B`.
7. Run `python web_app.py --host 127.0.0.1 --port 8200` locally.

## Local `.env` example
```env
LLM_PROVIDER=remote_openai
LLM_MODEL=LGAI-EXAONE/EXAONE-4.0-32B
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```

## Note
A fresh Colab runtime downloads the 32B model from the Hugging Face Hub. Keep enough CPU RAM while loading. The notebook verifies that no model layer is offloaded to CPU or disk before starting the API.


In [ ]:
# 1. Check GPU (A100 with at least 39GB VRAM required)
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required.")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}, VRAM: {gpu_memory_gb:.1f}GB")
if "A100" not in gpu_name or gpu_memory_gb < 39:
    raise RuntimeError("Select an A100 40GB or larger runtime. T4/L4 is not supported by this notebook.")


In [ ]:
# 2. Install dependencies
# EXAONE 4.0 is officially supported by Transformers 4.54.0 and newer.
# Gradio is not used by this FastAPI notebook and its preinstalled version
# requires huggingface_hub 1.x, while Transformers 4.x requires <1.0.
!pip uninstall -q -y gradio gradio_client
!pip install -q -U "transformers>=4.54.0,<5" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" "huggingface_hub>=0.34.0,<1.0" "requests==2.32.4" fastapi uvicorn pyngrok nest_asyncio

print("Dependencies installed. Continue to cell 3.")


In [ ]:
# 3. Check runtime
from importlib.metadata import version

import transformers
import torch

print("transformers", transformers.__version__)
print("accelerate", version("accelerate"))
print("bitsandbytes", version("bitsandbytes"))
print("huggingface_hub", version("huggingface_hub"))
print("requests", version("requests"))
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
# 4. Model source: download directly from the Hugging Face Hub inside Colab
HF_MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-32B"
MODEL_PATH = HF_MODEL_ID

print("MODEL_PATH=", MODEL_PATH)


In [ ]:
# 5. Hugging Face login
# Required to download the model directly from the Hugging Face Hub in this Colab runtime.
# Get a token from https://huggingface.co/settings/tokens
# If the model page requires accepting a license, accept it once at
# https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-32B before running this cell.
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token: ")
if hf_token.strip():
    login(token=hf_token.strip())
    print("Hugging Face login complete")
else:
    print("No token entered. Download may fail if the model requires authentication.")


In [ ]:
# 6. Load EXAONE model
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = MODEL_PATH

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

device_map = getattr(model, "hf_device_map", {})
offloaded = {name: device for name, device in device_map.items() if str(device) in {"cpu", "disk"}}
if offloaded:
    raise RuntimeError(f"Model layers were offloaded from the A100: {offloaded}")

print("loaded", MODEL_ID)
print("device", next(model.parameters()).device)
print("quantization", "NF4 4-bit")
print("model footprint GB", round(model.get_memory_footprint() / 1024**3, 2))
print("GPU allocated GB", round(torch.cuda.memory_allocated() / 1024**3, 2))


In [ ]:
# 7. Direct model test
def to_input_ids(encoded):
    if torch.is_tensor(encoded):
        return encoded.to(model.device)
    if hasattr(encoded, "get"):
        input_ids = encoded.get("input_ids")
        if input_ids is not None:
            return input_ids.to(model.device)
    raise TypeError(f"Unsupported tokenizer output: {type(encoded)!r}")

messages = [
    {"role": "system", "content": "Answer briefly in Korean."},
    {"role": "user", "content": "EXAONE server test. Reply in one Korean sentence."},
]

encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=False,
)
input_ids = to_input_ids(encoded)
attention_mask = torch.ones_like(input_ids, device=input_ids.device)

with torch.inference_mode():
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

new_tokens = output[0][input_ids.shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())


In [ ]:
# 8. Start OpenAI-compatible FastAPI server
import json
import threading
import traceback
import uuid

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

app = FastAPI(title="EXAONE 4.0 32B Colab Pro OpenAI-compatible API")
inference_lock = threading.Lock()
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 768

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict]
    temperature: float = 0.1
    max_tokens: int = Field(default=512, alias="max_tokens")
    stream: bool = False


def normalize_messages(messages: list[dict]) -> list[dict]:
    clean_messages = []
    for msg in messages:
        role = msg.get("role", "user")
        if role not in {"system", "user", "assistant"}:
            role = "user"
        clean_messages.append({"role": role, "content": str(msg.get("content", ""))})
    return clean_messages


def to_input_ids(encoded):
    if torch.is_tensor(encoded):
        return encoded.to(model.device)
    if hasattr(encoded, "get"):
        input_ids = encoded.get("input_ids")
        if input_ids is not None:
            return input_ids.to(model.device)
    raise TypeError(f"Unsupported tokenizer output: {type(encoded)!r}")


def build_model_inputs(messages: list[dict]) -> dict:
    clean_messages = normalize_messages(messages)
    try:
        encoded = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False,
        )
        input_ids = to_input_ids(encoded)
    except Exception:
        rendered = []
        for msg in clean_messages:
            rendered.append(f"[{msg['role']}]\n{msg['content']}")
        rendered.append("[assistant]\n")
        prompt = "\n\n".join(rendered)
        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)["input_ids"].to(model.device)

    if input_ids.shape[-1] > MAX_INPUT_TOKENS:
        input_ids = input_ids[:, -MAX_INPUT_TOKENS:]
    return {
        "input_ids": input_ids,
        "attention_mask": torch.ones_like(input_ids, device=input_ids.device),
    }


def generate_text(req: ChatRequest) -> str:
    # FastAPI handles sync requests in worker threads. Serialize generation so
    # concurrent browser requests cannot allocate multiple KV caches at once.
    with inference_lock:
        model_inputs = build_model_inputs(req.messages)
        input_length = model_inputs["input_ids"].shape[-1]
        output = None
        try:
            max_new_tokens = max(1, min(int(req.max_tokens or 512), MAX_NEW_TOKENS))
            generation_kwargs = {
                **model_inputs,
                "max_new_tokens": max_new_tokens,
                "do_sample": False,
                "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
                "eos_token_id": tokenizer.eos_token_id,
            }
            with torch.inference_mode():
                output = model.generate(**generation_kwargs)

            new_tokens = output[0][input_length:]
            return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        finally:
            del model_inputs
            if output is not None:
                del output
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_ID,
        "mode": "non-reasoning",
        "quantization": "NF4 4-bit",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "gpu_allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else 0,
    }


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}


@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    try:
        text = generate_text(req)
        completion_id = "chatcmpl-" + uuid.uuid4().hex

        if req.stream:
            def event_stream():
                chunk = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}],
                }
                yield "data: " + json.dumps(chunk, ensure_ascii=False) + "\n\n"
                done = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
                }
                yield "data: " + json.dumps(done, ensure_ascii=False) + "\n\n"
                yield "data: [DONE]\n\n"
            return StreamingResponse(event_stream(), media_type="text/event-stream")

        return {
            "id": completion_id,
            "object": "chat.completion",
            "model": req.model or MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }],
        }
    except Exception as exc:
        traceback.print_exc()
        is_memory_error = "not enough memory" in str(exc).lower() or "out of memory" in str(exc).lower()
        return JSONResponse(
            status_code=503 if is_memory_error else 500,
            content={
                "error": "ModelMemoryError" if is_memory_error else type(exc).__name__,
                "message": "모델 메모리가 부족합니다. 잠시 후 다시 시도하거나 Colab 런타임을 재시작하세요." if is_memory_error else str(exc),
            },
        )


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("server started: http://localhost:8000")


In [ ]:
# 9. Test local Colab API
import requests

payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "Reply in one Korean sentence. API test."}],
    "temperature": 0.1,
    "max_tokens": 128,
    "stream": False,
}

r = requests.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=180)
print(r.status_code)
print(r.text[:2000])


## Open ngrok public URL

Copy your token from https://dashboard.ngrok.com/get-started/your-authtoken.


In [ ]:
# 10. Open ngrok tunnel
from getpass import getpass
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

NGROK_TOKEN = "3EWgctUjXR3kFibnYJsoYBbboCC_6mvNunJ5fQQztjentPBRy"  # Optional: paste token here, but do not share the notebook with token saved.

if not NGROK_TOKEN.strip():
    NGROK_TOKEN = getpass("ngrok authtoken: ")

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN.strip())

try:
    public_url = ngrok.connect(8000).public_url
except PyngrokNgrokHTTPError as exc:
    print("ngrok failed:", exc)
    print("Close old ngrok endpoints in the ngrok dashboard and run this cell again.")
    raise

base_url = str(public_url).rstrip("/")
print("PUBLIC_URL=", base_url)
print("LLM_API_BASE=", base_url + "/v1")


In [ ]:
# 11. Test public ngrok URL
import requests

headers = {"ngrok-skip-browser-warning": "true"}

r = requests.get(base_url + "/health", headers=headers, timeout=30)
print("health", r.status_code, r.text)

r = requests.post(
    base_url + "/v1/chat/completions",
    headers=headers,
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "Reply in one Korean sentence. External connection test."}],
        "temperature": 0.1,
        "max_tokens": 128,
        "stream": False,
    },
    timeout=180,
)
print("chat", r.status_code)
print(r.text[:2000])


## Local PC setup

Put the printed `/v1` URL into `.env`.

```env
LLM_PROVIDER=remote_openai
LLM_MODEL=LGAI-EXAONE/EXAONE-4.0-32B
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```

Run locally:

```cmd
conda activate p311_ragreport
cd /d "C:\K-Safety Law RAG"
python web_app.py --host 127.0.0.1 --port 8000
```

CLI check:

```cmd
python cli.py
```
